# Ground-Motion Parameters (PGA / PGV / PGD / SA) on Google Colab

Runs SeisSol's official `ComputeGroundMotionParametersFromSurfaceOutput_Hybrid.py`
on a SeisSol **free-surface** output and plots the maps inline.

### Before you start — put the surface output on Google Drive
The GME tool only needs the *surface* product, so upload just these (keeps it small):
```
<your_output>/safs-surface.xdmf
<your_output>/safs-surface_cell/mesh0/      (connect.bin, v1.bin, v2.bin, ...)
<your_output>/safs-surface_vertex/mesh0/    (geometry.bin)
```
Set `DATA_DIR` (cell 4) to that folder. The notebook then **copies the surface
files to local disk** (cell 5) because reading big `.bin` files directly over the
Google Drive FUSE mount is very slow (it makes the run look stuck). Compute runs
on the fast local copy; results are copied back to Drive at the end.
(No macOS `fork` fix needed — Colab is Linux.)


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install Python dependencies
**scipy is pinned to 1.13.1 on purpose:** the deprecated `gmpe-smtk` imports
`scipy.integrate.cumtrapz`, which was **removed in scipy >= 1.14**. 1.13.1 still
has it (and the main GME script's `cumulative_trapezoid`).

In [ ]:
!pip -q install seissolxdmf seissolxdmfwriter "scipy==1.13.1" h5py lxml numpy
import scipy; print("scipy", scipy.__version__, "(needs cumtrapz -> must be <1.14)")

## 3. Fetch the GME script + clone gmpe-smtk
`gmpe-smtk` is a **mandatory** dependency and must sit next to the script
(the tool appends `<script_dir>/gmpe-smtk` to `sys.path`). We use the deprecated,
openquake-free commit.

In [ ]:
import os, urllib.request
GME_DIR = "/content/gme"
os.makedirs(GME_DIR, exist_ok=True)
SCRIPT = os.path.join(GME_DIR, "ComputeGroundMotionParametersFromSurfaceOutput_Hybrid.py")

URL = ("https://raw.githubusercontent.com/SeisSol/SeisSol/master/"
       "postprocessing/science/GroundMotionParametersMaps/"
       "ComputeGroundMotionParametersFromSurfaceOutput_Hybrid.py")
try:
    urllib.request.urlretrieve(URL, SCRIPT)
    assert "ComputeGroundMotionParameters" in open(SCRIPT).read()
    print("downloaded script ->", SCRIPT)
except Exception as e:
    print("AUTO-DOWNLOAD FAILED:", e)
    print("Upload your local copy of the .py to", GME_DIR, "and re-run this cell.")

if not os.path.isdir(os.path.join(GME_DIR, "gmpe-smtk")):
    !git clone -q https://github.com/GEMScienceTools/gmpe-smtk {GME_DIR}/gmpe-smtk
    !cd {GME_DIR}/gmpe-smtk && git checkout -q '4f008173e89f6e4ba4450fb43e95ffdf51a7c2ba^'
    print("cloned gmpe-smtk")
else:
    print("gmpe-smtk already present")

## 3b. (optional) Speed up — coarsen the GMRotD rotation
PGA/PGV/PGD/SA are **GMRotD50**: a median over fault-rotation angles, looped in pure
Python at **1 deg steps (90 angles per cell)** — this is the main cost (~10 min for
34k cells on 2 cores). Coarsening to **5 deg (18 angles)** is **~5x faster** and changes
RotD50 by **<1%**. Run this **after cell 3** (which re-downloads the script each session).
Set `SPEEDUP=False` to keep the exact 1 deg default.

In [ ]:
SPEEDUP = True   # 1 deg -> 5 deg rotation (90 -> 18 angles); ~5x faster, RotD50 within ~1%
if SPEEDUP:
    s = open(SCRIPT).read()
    s2 = s.replace("np.arange(0., 90., 1.)", "np.arange(0., 90., 5.)")
    open(SCRIPT, "w").write(s2)
    print("rotation angle step -> 5 deg" if s != s2 else "(pattern not found / already patched)")
else:
    print("keeping exact 1 deg rotation")

## 4. Point at your data on Drive
**Edit `DATA_DIR`** to the folder that holds `safs-surface.xdmf`.

In [ ]:
DATA_DIR = "/content/drive/MyDrive/seisol_quakeworx/output_safs_v2.2.0_constant_mat_onfaultpoints"  # <-- EDIT

XDMF = os.path.join(DATA_DIR, "safs-surface.xdmf")
OUT_DIR = DATA_DIR
assert os.path.exists(XDMF), f"{XDMF} not found - fix DATA_DIR"
print("Drive input:", XDMF)

## 5. Copy the surface files to local disk (fast I/O) — recommended
Reading the `.bin` files straight off the Drive FUSE mount is slow and makes the
run look frozen. Copy them to `/content` (fast local disk) and run there instead.
Skip this cell only if your data is already on local disk.

In [ ]:
import shutil, time
LOCAL = "/content/gme_data"
os.makedirs(LOCAL, exist_ok=True)
t0 = time.time()
for item in ("safs-surface.xdmf", "safs-surface_cell", "safs-surface_vertex"):
    src, dst = os.path.join(DATA_DIR, item), os.path.join(LOCAL, item)
    if not os.path.exists(src):
        raise FileNotFoundError(src)
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dst)
XDMF, OUT_DIR = os.path.join(LOCAL, "safs-surface.xdmf"), LOCAL   # run on the fast local copy
print(f"copied surface files to {LOCAL} in {time.time()-t0:.0f}s")
print("XDMF   =", XDMF)
print("OUT_DIR=", OUT_DIR)

## 6. Run the GME tool
PGA/PGV/PGD = GMRotD50. Output (`safs-GME-surface.*`) is written to `OUT_DIR`.

- **`PERIODS`** controls SA(T) cost — fewer periods = much faster. PGA/PGV/PGD are
  always computed regardless. `[]` -> default 11 periods (slowest).
- Output **streams live** (unbuffered): you'll see `Waiting for N tasks...` count
  down every ~10 s. If it shows no output for a long time before
  `done reading surface data`, you skipped cell 5 (Drive I/O is the bottleneck).
- **`XDMF` is placed before `--periods`** because `--periods` uses `nargs='+'`
  and would otherwise swallow the file path (argparse exit code 2).

In [ ]:
import sys, subprocess, time
env = dict(os.environ, PYTHONUNBUFFERED="1")     # force the child to flush its prints
MP  = str(os.cpu_count() or 1)                   # use all cores (free Colab = 2)
PERIODS = ["2", "5"]                             # SA periods; fewer = faster. [] = default 11.

cmd = [sys.executable, "-u", SCRIPT, XDMF, "--noMPI", "--MP", MP]
if PERIODS:
    cmd += ["--periods", *PERIODS]

print("running:", " ".join(cmd), "\n", flush=True)
t0 = time.time()
proc = subprocess.Popen(cmd, cwd=OUT_DIR, env=env, text=True, bufsize=1,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
for line in proc.stdout:                         # print each line as it arrives
    print(f"[{time.time()-t0:5.0f}s] {line}", end="", flush=True)
proc.wait()
assert proc.returncode == 0, f"GME tool failed (exit {proc.returncode}) - see output above"
print(f"\nDONE in {time.time()-t0:.0f}s -> {os.path.join(OUT_DIR, 'safs-GME-surface.xdmf')}")

## 7. Copy results back to Drive
`/content` is wiped when the runtime disconnects, so persist the GME output to Drive.

In [ ]:
import shutil
for f in ("safs-GME-surface.xdmf", "safs-GME-surface.h5"):
    src = os.path.join(OUT_DIR, f)
    if os.path.exists(src) and os.path.abspath(OUT_DIR) != os.path.abspath(DATA_DIR):
        shutil.copy2(src, os.path.join(DATA_DIR, f))
        print("copied to Drive:", f)
print("done (results now persist in", DATA_DIR + ")")

## 8. Inspect the result (value ranges)

In [ ]:
import h5py, numpy as np
H5 = os.path.join(OUT_DIR, "safs-GME-surface.h5")
with h5py.File(H5, "r") as f:
    print("datasets:", list(f.keys()), "\n")
    for k in ("PGA", "PGV", "PGD"):
        a = f[k][:]
        print(f"{k:4s} min={a.min():.4g}  max={a.max():.4g}  mean={a.mean():.4g}")
print("\nUnits: PGA m/s^2 (/9.80665 -> g), PGV m/s, PGD m. Values are GMRotD50 (horizontal).")

## 9. Plot PGV / PGD / PGA maps inline
Colormap is clipped at the 99th percentile so the on-fault-trace extremes
(which are fault slip, not off-fault shaking) don't wash out the map.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import numpy as np, h5py

with h5py.File(H5, "r") as f:
    geom = f["geometry"][:]; conn = f["connect"][:]
    fields = {k: f[k][:].ravel() for k in ("PGV", "PGD", "PGA")}  # GME fields are (1, ncells) -> ravel to (ncells,)

triang = mtri.Triangulation(geom[:, 0], geom[:, 1], conn)
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for ax, (name, unit) in zip(axes, [("PGV", "m/s"), ("PGD", "m"), ("PGA", "m/s^2")]):
    val = fields[name]
    tpc = ax.tripcolor(triang, facecolors=val, cmap="inferno", shading="flat")
    tpc.set_clim(0, np.percentile(val, 99))
    ax.set_aspect("equal"); ax.set_title(f"{name}  [{unit}]")
    ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")
    fig.colorbar(tpc, ax=ax, shrink=0.85)
plt.tight_layout(); plt.show()

## Notes / caveats
- **dt = 0.5 s surface sampling** -> **PGA and short-period SA are aliased**; trust PGV/PGD.
  For true PGA use the high-rate receiver `.dat` files instead.
- On-fault-trace cells record fault slip (huge values), not off-fault ground motion.
- Another case: change `DATA_DIR` (cell 4), re-run cells 4-9.
- `module smtk not found` + `cannot import name 'cumtrapz'` -> scipy too new; re-run cell 2.
- Looks stuck with no output -> you skipped cell 5 (copy to local); Drive reads are slow.
